# Unpack — Domain-Specific Knowledge Graph (DSKG)
### Complete Python Implementation for Google Colab

This notebook provides a complete implementation of the **Unpack DSKG**, replicating the Go-based neuro-symbolic anchor. It loads a two-layer Knowledge Graph (T-Box and A-Box) onto Neo4j.

**ACADEMIC FRAMING (Hogan et al. 2021):**
*   **T-Box (Terminology Box):** Formal ontology (Class, Relation, Property, Curriculum).
*   **A-Box (Assertion Box):** Instance data (Concepts, Formulas, L2Labels, Videos, UseCases).


### 1. Install Dependencies and Import Libraries
Install the `neo4j` python driver and `pandas` for data handling.


In [40]:
!pip install neo4j pandas --quiet

import os
import time
import pandas as pd
from datetime import datetime
from dataclasses import dataclass
from typing import List, Optional, Dict, Any
from neo4j import GraphDatabase, Driver

# Configuration for Colab
DATA_DIR = "./data/raw" # Update this to your Google Drive path if needed
NEO4J_URI=""
NEO4J_USERNAME=""
NEO4J_PASSWORD=""
NEO4J_DATABASE=""
AURA_INSTANCEID=""
AURA_INSTANCENAME=""


### 2. Configure Neo4j Connection
Configure the Neo4j driver using credentials.


In [41]:
def get_driver(uri, user, pwd):
    driver = GraphDatabase.driver(uri, auth=(user, pwd))
    try:
        driver.verify_connectivity()
        print(f"✓ Connected to Neo4j at {uri}")
        return driver
    except Exception as e:
        print(f"✗ Connection failed: {e}")
        return None

# Use environment variables if available
uri = NEO4J_URI
user = NEO4J_USERNAME
pwd = NEO4J_PASSWORD

driver = get_driver(uri, user, pwd)

def clear_all(tx):
    tx.run("MATCH ()-[r]-() DELETE r")
    tx.run("MATCH (n) DELETE n")
    print("🧹 Cleared all data")

✓ Connected to Neo4j at neo4j+s://86b769c2.databases.neo4j.io


### 3. Initialize Database Schema (Constraints and Indexes)
Create uniqueness constraints and full-text indexes for efficient retrieval.


In [42]:
def create_schema(tx):
    statements = [
        "CREATE CONSTRAINT concept_id_unique IF NOT EXISTS FOR (c:Concept) REQUIRE c.id IS UNIQUE",
        "CREATE CONSTRAINT formula_id_unique IF NOT EXISTS FOR (f:Formula) REQUIRE f.id IS UNIQUE",
        "CREATE CONSTRAINT l2label_id_unique IF NOT EXISTS FOR (l:L2Label) REQUIRE l.id IS UNIQUE",
        "CREATE CONSTRAINT video_id_unique IF NOT EXISTS FOR (v:VideoResource) REQUIRE v.id IS UNIQUE",
        "CREATE CONSTRAINT usecase_id_unique IF NOT EXISTS FOR (u:UseCase) REQUIRE u.id IS UNIQUE",
        "CREATE CONSTRAINT class_name_unique IF NOT EXISTS FOR (c:Class) REQUIRE c.name IS UNIQUE",
        "CREATE CONSTRAINT relation_name_uniq IF NOT EXISTS FOR (r:Relation) REQUIRE r.name IS UNIQUE",
        "CREATE CONSTRAINT property_name_uniq IF NOT EXISTS FOR (p:Property) REQUIRE p.name IS UNIQUE",
        "CREATE CONSTRAINT curriculum_id_uniq IF NOT EXISTS FOR (c:Curriculum) REQUIRE c.id IS UNIQUE",
        "CREATE INDEX concept_type IF NOT EXISTS FOR (c:Concept) ON (c.type)",
        "CREATE INDEX concept_layer IF NOT EXISTS FOR (c:Concept) ON (c.curriculum_layer)",
        "CREATE INDEX l2label_lang IF NOT EXISTS FOR (l:L2Label) ON (l.language_code)",
        "CREATE INDEX video_lang IF NOT EXISTS FOR (v:VideoResource) ON (v.language)",
        "CREATE INDEX video_diff IF NOT EXISTS FOR (v:VideoResource) ON (v.difficulty)",
        "CREATE INDEX formula_primary IF NOT EXISTS FOR (f:Formula) ON (f.is_primary)",
        "CREATE INDEX formula_level IF NOT EXISTS FOR (f:Formula) ON (f.display_level)",
        "CREATE INDEX usecase_domain IF NOT EXISTS FOR (u:UseCase) ON (u.domain)",
        "CREATE FULLTEXT INDEX concept_search IF NOT EXISTS FOR (c:Concept) ON EACH [c.name, c.description, c.core_theory]",
        "CREATE FULLTEXT INDEX l2label_search IF NOT EXISTS FOR (l:L2Label) ON EACH [l.label, l.description_l2, l.curriculum_name]",
        "CREATE FULLTEXT INDEX usecase_search IF NOT EXISTS FOR (u:UseCase) ON EACH [u.description, u.problem_example, u.domain]"
    ]
    for stmt in statements:
        tx.run(stmt)
    print("✓ Schema initialized: 9 constraints, 8 indexes, 3 fulltext indexes")


### 4. Load T-Box (Ontology Layer)
Load the formal ontology layer, defining Classes, Relations, and Properties.


In [43]:
def load_ontology(tx):
    classes = [
        {"name": "Concept", "description": "A mathematical concept at undergraduate level.", "scope": "undergraduate mathematics", "valid_types": ["Foundation", "Calculus", "Geometry", "Rate of Change", "Goal", "Equation Type"]},
        {"name": "Formula", "description": "A formula node. Separate so UI agent fetches only formula on hover without loading full theory.", "scope": "mathematical notation"},
        {"name": "L2Label", "description": "A canonical multilingual label. Separate so multilingual agent queries by language_code without scanning all concept properties.", "scope": "multilingual localisation"},
        {"name": "VideoResource", "description": "An educational YouTube video. Separate so Agent 5 filters by language and difficulty independently of concept data.", "scope": "educational media"},
        {"name": "UseCase", "description": "A real-world application. Answers: why am I learning this? Shown in hover state 3.", "scope": "contextual scaffolding"},
        {"name": "ProblemType", "description": "A category of mathematics word problem (e.g. Related Rates). Links to Concept via EXEMPLIFIES and INVOLVES.", "scope": "problem classification"},
        {"name": "Curriculum", "description": "Academic curriculum providing scope context for the graph.", "scope": "academic scope"},
    ]
    for c in classes:
        tx.run("MERGE (n:Class {name: $p.name}) SET n = $p", p=c)

    relations = [
        {"name": "REQUIRES", "domain": "Concept", "range": "Concept", "definition": "Student cannot engage with target without mastering source. Asymmetric, transitive, weighted.", "is_transitive": True, "is_symmetric": False, "weight_meaning": "1.0=hard prerequisite, 0.8=recommended, 0.7=helpful"},
        {"name": "RELATED_TO", "domain": "Concept", "range": "Concept", "definition": "Two concepts co-occur in same problem or share mathematical structure. Neither is prerequisite.", "is_transitive": False, "is_symmetric": True, "weight_meaning": "1.0=definitionally linked, 0.8=frequently combined"},
        {"name": "HAS_FORMULA", "domain": "Concept", "range": "Formula", "definition": "Concept has a formula. Primary formula shown at hover state 1 (low CL).", "is_transitive": False, "is_symmetric": False},
        {"name": "HAS_LABEL", "domain": "Concept", "range": "L2Label", "definition": "Concept has canonical name in a specific language. Multilingual Agent B1 queries by language_code for textbook-canonical term.", "is_transitive": False, "is_symmetric": False},
        {"name": "HAS_RESOURCE", "domain": "Concept", "range": "VideoResource", "definition": "Concept has an educational video. Agent 5 attaches videos. UI queries by language for L1 content.", "is_transitive": False, "is_symmetric": False},
        {"name": "APPLIED_IN", "domain": "Concept", "range": "UseCase", "definition": "Concept applied in a real-world use case. Shown in hover state 3 (high support).", "is_transitive": False, "is_symmetric": False},
        {"name": "EXEMPLIFIES", "domain": "ProblemType", "range": "Concept", "definition": "Word problem of this type tests this Concept as the primary operation.", "is_transitive": False, "is_symmetric": False},
        {"name": "INVOLVES", "domain": "ProblemType", "range": "Concept", "definition": "Word problem of this type uses this Concept as a supporting operation.", "is_transitive": False, "is_symmetric": False},
        {"name": "COVERS", "domain": "Curriculum", "range": "Concept", "definition": "This Curriculum includes this Concept. Enables scope-bounded KG queries.", "is_transitive": False, "is_symmetric": False},
    ]
    for r in relations:
        tx.run("MERGE (n:Relation {name: $p.name}) SET n = $p", p=r)

    properties = [
        {"name": "id", "applies_to": "Concept", "datatype": "String", "definition": "Unique stable slug. Used as graph_node_id in Unpack UI JSON."},
        {"name": "core_theory", "applies_to": "Concept", "datatype": "String", "definition": "Strict mathematical definition shown at hover state 2 (medium CL)."},
        {"name": "type", "applies_to": "Concept", "datatype": "String", "definition": "Pedagogical category controlling UI token colour.", "valid_values": ["Foundation", "Calculus", "Geometry", "Rate of Change", "Goal", "Equation Type"]},
        {"name": "color", "applies_to": "Concept", "datatype": "String", "definition": "Hex colour for UI token underline and tooltip badge."},
        {"name": "curriculum_layer", "applies_to": "Concept", "datatype": "Integer", "definition": "0=pre-calc, 1=differentiation, 2=integration, 3=series, 4=multivariable."},
        {"name": "latex", "applies_to": "Formula", "datatype": "String", "definition": "Raw LaTeX rendered by KaTeX or MathJax in the progressive disclosure UI."},
        {"name": "notation_plain", "applies_to": "Formula", "datatype": "String", "definition": "ASCII plain-text formula shown when LaTeX rendering unavailable."},
        {"name": "display_level", "applies_to": "Formula", "datatype": "Integer", "definition": "1=hover (low CL), 2=click (medium CL)."},
        {"name": "language_code", "applies_to": "L2Label", "datatype": "String", "definition": "ISO 639-1 code. Multilingual Agent B1 queries on this field."},
        {"name": "text_direction", "applies_to": "L2Label", "datatype": "String", "definition": "ltr or rtl. UI sets CSS direction from this. Critical for Arabic."},
        {"name": "curriculum_name", "applies_to": "L2Label", "datatype": "String", "definition": "Textbook-canonical term in student language. Not a translation — a proper curriculum name."},
        {"name": "url", "applies_to": "VideoResource", "datatype": "String", "definition": "Direct YouTube video URL."},
        {"name": "rank_score", "applies_to": "VideoResource", "datatype": "Float", "definition": "Agent 5 semantic rank: channel authority × transcript similarity × duration filter."},
    ]
    for p in properties:
        tx.run("MERGE (n:Property {name: $p.name}) SET n = $p", p=p)

    curriculum = {
        "id": "undergrad_calculus",
        "name": "Undergraduate Calculus",
        "scope": "First and second year university mathematics",
        "covers_layers": [0, 1, 2, 3, 4],
        "study_context": "Unpack RQ3 — non-native English speaking STEM students",
        "citation": "Hogan et al. (2021). Knowledge Graphs. ACM Computing Surveys 54(4)."
    }
    tx.run("MERGE (cur:Curriculum {id: $id}) SET cur += $p", id=curriculum["id"], p=curriculum)
    print("✓ Ontology (T-Box) loaded: 7 Class, 9 Relation, 13 Property, 1 Curriculum")


### 5. Load A-Box (Instance Data: Concepts, Formulas, L2Labels, Videos, UseCases)
Batch upload instance data from CSV files and establish relationships.


In [44]:
def load_concepts(tx, concepts_df):
    for _, row in concepts_df.iterrows():
        tx.run("""
            MERGE (c:Concept {id:$id})
            SET c.name=$name, c.description=$desc, c.core_theory=$theory,
                c.type=$type, c.color=$color, c.curriculum_layer=$layer,
                c.updated_at=datetime()
            """, id=row['concept_id'], name=row['concept_name'], desc=row['description'],
               theory=row['core_theory'], type=row['type'], color=row['color'], layer=int(row['curriculum_layer']))
    print(f"   ✓ {len(concepts_df)} Concept nodes")

def load_formulas(tx, formulas_df):
    for _, row in formulas_df.iterrows():
        tx.run("""
            MATCH (c:Concept {id:$cid})
            MERGE (f:Formula {id:$id})
            SET f.name=$name, f.latex=$latex, f.notation_plain=$plain,
                f.display_level=$level, f.is_primary=$primary
            MERGE (c)-[:HAS_FORMULA {primary:$primary}]->(f)
            """, cid=row['concept_id'], id=row['formula_id'], name=row['formula_name'],
               latex=row['latex'], plain=row['notation_plain'],
               level=int(row['display_level']), primary=str(row['is_primary']).lower() == 'true')
    print(f"   ✓ {len(formulas_df)} Formula nodes + HAS_FORMULA edges")

def load_l2_labels(tx, labels_df):
    for _, row in labels_df.iterrows():
        tx.run("""
            MATCH (c:Concept {id:$cid})
            MERGE (l:L2Label {id:$id})
            SET l.language_code=$lang, l.label=$label, l.description_l2=$desc,
                l.curriculum_name=$curriculum, l.text_direction=$dir
            MERGE (c)-[:HAS_LABEL]->(l)
            """, cid=row['concept_id'], id=row['label_id'], lang=row['language_code'],
               label=row['label'], desc=row['description_l2'],
               curriculum=row['curriculum_name'], dir=row['text_direction'])
    print(f"   ✓ {len(labels_df)} L2Label nodes + HAS_LABEL edges")

def load_videos(tx, videos_df):
    for _, row in videos_df.iterrows():
        tx.run("""
            MATCH (c:Concept {id:$cid})
            MERGE (v:VideoResource {id:$id})
            SET v.platform=$platform, v.title=$title, v.url=$url,
                v.language=$lang, v.duration_sec=$dur, v.difficulty=$diff, v.channel=$ch
            MERGE (c)-[:HAS_RESOURCE]->(v)
            """, cid=row['concept_id'], id=row['video_id'], platform=row['platform'],
               title=row['title'], url=row['url'], lang=row['language'],
               dur=int(row['duration_sec']), diff=row['difficulty'], ch=row['channel'])
    print(f"   ✓ {len(videos_df)} VideoResource nodes + HAS_RESOURCE edges")

def load_use_cases(tx, use_cases_df):
    for _, row in use_cases_df.iterrows():
        tx.run("""
            MATCH (c:Concept {id:$cid})
            MERGE (u:UseCase {id:$id})
            SET u.domain=$domain, u.description=$desc, u.problem_example=$example
            MERGE (c)-[:APPLIED_IN]->(u)
            """, cid=row['concept_id'], id=row['usecase_id'], domain=row['domain'],
               desc=row['description'], example=row['problem_example'])
    print(f"   ✓ {len(use_cases_df)} UseCase nodes + APPLIED_IN edges")

def load_edges(tx, edges_df):
    req, rel = 0, 0
    for _, row in edges_df.iterrows():
        etype = row['relationship_type']
        q = f"""
            MATCH (s:Concept {{id:$sid}}), (t:Concept {{id:$tid}})
            MERGE (s)-[r:{etype}]->(t)
            SET r.weight=$w, r.updated_at=datetime()
            """
        tx.run(q, sid=row['source_id'], tid=row['target_id'], w=float(row['weight']))
        if etype == 'REQUIRES': req += 1
        elif etype == 'RELATED_TO': rel += 1
    print(f"   ✓ {req} REQUIRES + {rel} RELATED_TO edges")

def link_curriculum(tx):
    res = tx.run("""
        MATCH (cur:Curriculum {id:"undergrad_calculus"}), (c:Concept)
        MERGE (cur)-[:COVERS {layer: c.curriculum_layer}]->(c)
        RETURN count(*) as n
        """)
    n = res.single()['n']
    print(f"   ✓ {n} COVERS edges linked to curriculum")


### 6. Verification Suite (Agent Access Pattern Tests)
Run 11 verification tests to ensure the Knowledge Graph is correctly loaded and supports various agent access patterns.


In [45]:
def run_verification(tx):
    print("\n" + "="*62)
    print("  Verification — 11 tests across all agent access patterns")
    print("="*62)

    tests = [
        ("T01 T-Box: Class nodes ≥ 7", "MATCH (c:Class) RETURN count(c) as n", lambda r: r[0]['n'] >= 7),
        ("T02 T-Box: Relation nodes ≥ 9", "MATCH (r:Relation) RETURN count(r) as n", lambda r: r[0]['n'] >= 9),
        ("T03 T-Box: Property nodes ≥ 13", "MATCH (p:Property) RETURN count(p) as n", lambda r: r[0]['n'] >= 13),
        ("T04 A-Box: 39 Concept nodes", "MATCH (c:Concept) RETURN count(c) as n", lambda r: r[0]['n'] >= 39),
        ("T05 Prereq chain for related_rates", 'MATCH (:Concept{id:"related_rates"})-[:REQUIRES*1..6]->(p:Concept) RETURN count(distinct p) as n', lambda r: r[0]['n'] >= 7),
        ("T06 UI Agent: chain_rule primary formula", 'MATCH (:Concept{id:"chain_rule"})-[:HAS_FORMULA]->(f:Formula) WHERE f.is_primary=true AND f.display_level=1 RETURN f.notation_plain as f', lambda r: len(r) >= 1),
        ("T07 Multilingual: Sinhala for chain_rule", 'MATCH (:Concept{id:"chain_rule"})-[:HAS_LABEL]->(l:L2Label{language_code:"si"}) RETURN l.label as l', lambda r: len(r) >= 1),
        ("T08 Multilingual: Arabic direction is rtl", 'MATCH (:Concept{id:"related_rates"})-[:HAS_LABEL]->(l:L2Label{language_code:"ar"}) RETURN l.text_direction as d', lambda r: r[0]['d'] == "rtl"),
        ("T09 Agent 5: EN video for related_rates", 'MATCH (:Concept{id:"related_rates"})-[:HAS_RESOURCE]->(v:VideoResource{language:"en"}) RETURN v.title as t', lambda r: len(r) >= 1),
        ("T10 Context Agent: use case for optimization", 'MATCH (:Concept{id:"optimization"})-[:APPLIED_IN]->(u:UseCase) RETURN u.domain as d', lambda r: len(r) >= 1),
        ("T11 Curriculum COVERS all 39 Concepts", 'MATCH (:Curriculum{id:"undergrad_calculus"})-[:COVERS]->(c:Concept) RETURN count(c) as n', lambda r: r[0]['n'] >= 39),
    ]

    pass_count, fail_count = 0, 0
    for label, query, check in tests:
        try:
            res = list(tx.run(query))
            if check(res):
                print(f"  ✓ {label:48} {len(res)} row(s)")
                pass_count += 1
            else:
                print(f"  ✗ {label:48} FAILED")
                fail_count += 1
        except Exception as e:
            print(f"  ✗ {label:48} ERROR: {e}")
            fail_count += 1

    print("\n" + f"  Result: {pass_count} passed / {pass_count+fail_count} total")
    print("="*62)


### 7. Execution: Load Data and Run Tests
Run the loading functions in order.


In [46]:
if driver:
    # 1. Read CSV Data
    try:
        concepts_df = pd.read_csv(f"{DATA_DIR}/concepts.csv")
        formulas_df = pd.read_csv(f"{DATA_DIR}/formulas.csv")
        labels_df = pd.read_csv(f"{DATA_DIR}/l2_labels.csv")
        videos_df = pd.read_csv(f"{DATA_DIR}/videos.csv")
        use_cases_df = pd.read_csv(f"{DATA_DIR}/use_cases.csv")
        # Rename column to match expected key in load_use_cases function
        use_cases_df.rename(columns={'use_case_id': 'usecase_id'}, inplace=True)
        edges_df = pd.read_csv(f"{DATA_DIR}/edges.csv")
        print("✓ CSV files read from " + DATA_DIR)
    except FileNotFoundError as e:
        print(f"✗ Files not found. Ensure CSVs are in {DATA_DIR}: {e}")
        # Stop execution
        raise e

    # 2. Run Database Operations
    with driver.session() as session:
        # Clear database
        session.execute_write(clear_all)

        start_time = time.time()

        # Step 1: Schema
        print("\nStep 1/9  Schema — constraints + indexes")
        session.execute_write(create_schema)

        # Step 2: Ontology (T-Box)
        print("Step 2/9  Ontology — T-Box (Class, Relation, Property, Curriculum)")
        session.execute_write(load_ontology)

        # A-Box Loading
        print("Step 3/9  Concepts — 39 :Concept nodes")
        session.execute_write(load_concepts, concepts_df)

        print("Step 4/9  Formulas — :Formula + HAS_FORMULA")
        session.execute_write(load_formulas, formulas_df)

        print("Step 5/9  L2 Labels — :L2Label + HAS_LABEL")
        session.execute_write(load_l2_labels, labels_df)

        print("Step 6/9  Videos — :VideoResource + HAS_RESOURCE")
        session.execute_write(load_videos, videos_df)

        print("Step 7/9  Use Cases — :UseCase + APPLIED_IN")
        session.execute_write(load_use_cases, use_cases_df)

        print("Step 8/9  Edges — REQUIRES + RELATED_TO")
        session.execute_write(load_edges, edges_df)

        print("Step 9/9  Curriculum — COVERS edges to all :Concept")
        session.execute_write(link_curriculum)

        elapsed = time.time() - start_time
        print("\n" + "="*62)
        print("  Unpack DSKG — Build Complete")
        print(f"  Build time: {elapsed:.2f} seconds")
        print("="*62)

        # Final Verification
        session.execute_read(run_verification)

    # Removed driver.close() to keep the connection open for further queries
else:
    print("Driver not initialized. Please verify your credentials and connection.")

✓ CSV files read from ./data/raw
🧹 Cleared all data

Step 1/9  Schema — constraints + indexes
✓ Schema initialized: 9 constraints, 8 indexes, 3 fulltext indexes
Step 2/9  Ontology — T-Box (Class, Relation, Property, Curriculum)
✓ Ontology (T-Box) loaded: 7 Class, 9 Relation, 13 Property, 1 Curriculum
Step 3/9  Concepts — 39 :Concept nodes
   ✓ 39 Concept nodes
Step 4/9  Formulas — :Formula + HAS_FORMULA
   ✓ 58 Formula nodes + HAS_FORMULA edges
Step 5/9  L2 Labels — :L2Label + HAS_LABEL
   ✓ 156 L2Label nodes + HAS_LABEL edges
Step 6/9  Videos — :VideoResource + HAS_RESOURCE
   ✓ 52 VideoResource nodes + HAS_RESOURCE edges
Step 7/9  Use Cases — :UseCase + APPLIED_IN
   ✓ 45 UseCase nodes + APPLIED_IN edges
Step 8/9  Edges — REQUIRES + RELATED_TO
   ✓ 64 REQUIRES + 16 RELATED_TO edges
Step 9/9  Curriculum — COVERS edges to all :Concept
   ✓ 39 COVERS edges linked to curriculum

  Unpack DSKG — Build Complete
  Build time: 2.98 seconds

  Verification — 11 tests across all agent access p

In [47]:
if driver:
    with driver.session() as session:
        # Example Cypher query: Count all nodes
        result = session.run("MATCH (n) RETURN count(n) AS node_count")
        node_count = result.single()["node_count"]
        print(f"Total number of nodes in the graph: {node_count}")

        # Example Cypher query: List all Concept names
        print("\n--- First 5 Concept Names ---")
        concept_names_result = session.run("MATCH (c:Concept) RETURN c.name AS conceptName LIMIT 5")
        for record in concept_names_result:
            print(record["conceptName"])
else:
    print("Driver not initialized. Please connect to Neo4j first.")

Total number of nodes in the graph: 380

--- First 5 Concept Names ---
Arithmetic
Algebra
Coordinate Geometry
Pythagorean Theorem
Similar Triangles


// ═══════════════════════════════════════════════════════════════════════════════
// Unpack Knowledge Graph — 06_agent_queries.cypher
// ───────────────────────────────────────────────────────────────────────────────
// PURPOSE : Reference library of all parameterised Cypher queries used by the
//           ADK agent tools. This is NOT a migration script — it is documentation
//           and a test suite. Run each query manually in Neo4j Browser to verify
//           the graph is correctly answering each agent's access pattern.
//
// PARAMETERS: shown as $param_name — substitute real values when testing.
//
// AGENT MAPPING:
//   Agent 1 (Lexical Parser)       → SECTION 1: Fulltext search
//   Agent 2 (Symbolic Router)      → SECTION 2: Prerequisite traversal
//   Agent 3 (Gap Detector)         → SECTION 3: Null-token near-match search
//   Agent 4 (Graph Mutator)        → SECTION 4: MERGE new concept (HITL)
//   Agent 5 (Resource Curator)     → SECTION 5: Video attachment
//   UI Mapping Agent (Phase 5)     → SECTION 6: Progressive disclosure queries
//   Multilingual Agent (B1–B4)     → SECTION 7: L2 localisation queries
// ═══════════════════════════════════════════════════════════════════════════════


// ════════════════════════════════════════════════════════════════════════════════
// SECTION 1 — AGENT 1: Lexical Parser
// Tool: kg_verify(token)
// Purpose: Check whether a token maps to a known concept in the KG.
//          Returns the concept if found, null if not (triggers Agent 3).
// ════════════════════════════════════════════════════════════════════════════════

// 1a. Exact ID match (fastest — used when Agent 1 is confident)
MATCH (c:Concept {id: $concept_id})
RETURN c.id, c.name, c.type, c.color, c.curriculum_layer;

// 1b. Fulltext name search (used when token is a natural language phrase)
CALL db.index.fulltext.queryNodes("concept_search", $search_query)
YIELD node, score
WHERE score > 1.0
RETURN node.id AS concept_id, node.name AS name, node.type AS type, score
ORDER BY score DESC
LIMIT 5;

// 1c. L2 label search (when student input is in native language)
CALL db.index.fulltext.queryNodes("l2label_search", $native_term)
YIELD node AS label_node, score
MATCH (c:Concept)-[:HAS_LABEL]->(label_node)
RETURN c.id AS concept_id, c.name AS name, label_node.language_code AS lang,
       label_node.label AS native_label, score
ORDER BY score DESC
LIMIT 5;


// ════════════════════════════════════════════════════════════════════════════════
// SECTION 2 — AGENT 2: Symbolic Router
// Tool: kg_prereqs(concept_id, depth)
// Purpose: Traverse the REQUIRES chain to build the prerequisite mind map.
//          Returns the full chain up to $depth hops.
// ════════════════════════════════════════════════════════════════════════════════

// 2a. Direct prerequisites only (depth = 1)
MATCH (c:Concept {id: $concept_id})-[:REQUIRES]->(p:Concept)
RETURN p.id, p.name, p.type, p.color, p.curriculum_layer
ORDER BY p.curriculum_layer;

// 2b. Full transitive chain (depth 1–4, the core Phase 4 query)
MATCH (c:Concept {id: $concept_id})-[:REQUIRES*1..4]->(p:Concept)
RETURN DISTINCT p.id      AS prereq_id,
                p.name    AS prereq_name,
                p.type    AS prereq_type,
                p.color   AS prereq_color,
                p.curriculum_layer AS layer
ORDER BY layer, prereq_name;

// 2c. Full prerequisite chain with path length (for mind map depth colouring)
MATCH path = (c:Concept {id: $concept_id})-[:REQUIRES*1..6]->(p:Concept)
RETURN DISTINCT p.id AS prereq_id,
                p.name AS prereq_name,
                min(length(path)) AS min_hops,
                p.curriculum_layer AS layer
ORDER BY min_hops, layer;

// 2d. Related concepts (RELATED_TO, for secondary mind map connections)
MATCH (c:Concept {id: $concept_id})-[:RELATED_TO]->(r:Concept)
RETURN r.id, r.name, r.type, r.color
ORDER BY r.name;

// 2e. Full context query (all connected nodes for one concept — Phase 4 complete)
MATCH (c:Concept {id: $concept_id})
OPTIONAL MATCH (c)-[:REQUIRES*1..4]->(prereq:Concept)
OPTIONAL MATCH (c)-[:RELATED_TO]->(related:Concept)
OPTIONAL MATCH (c)-[:HAS_FORMULA {primary: true}]->(formula:Formula)
OPTIONAL MATCH (c)-[:HAS_LABEL]->(label:L2Label {language_code: $lang})
RETURN
  c.id              AS concept_id,
  c.name            AS concept_name,
  c.core_theory     AS theory,
  c.type            AS type,
  c.color           AS color,
  collect(DISTINCT prereq.id)      AS prerequisites,
  collect(DISTINCT related.id)     AS related_concepts,
  formula.notation_plain           AS primary_formula,
  label.label                      AS native_label,
  label.text_direction             AS text_direction;


// ════════════════════════════════════════════════════════════════════════════════
// SECTION 3 — AGENT 3: Gap Detector
// Tool: kg_near_match(token)
// Purpose: When a token returns null from Agent 2, Agent 3 checks if a
//          similar concept exists (near-miss) before proposing a new node.
// ════════════════════════════════════════════════════════════════════════════════

// 3a. Near-miss fulltext search (returns candidates with confidence scores)
CALL db.index.fulltext.queryNodes("concept_search", $unresolved_token)
YIELD node, score
RETURN node.id AS candidate_id, node.name AS candidate_name, score
ORDER BY score DESC
LIMIT 3;

// 3b. Check if a proposed new concept ID already exists
MATCH (c:Concept {id: $proposed_id})
RETURN c.id, c.name;

// 3c. Find the most appropriate prerequisite for a proposed new concept
// (Used to wire new nodes correctly into the graph)
MATCH (c:Concept)
WHERE c.curriculum_layer = $proposed_layer - 1
RETURN c.id, c.name, c.type
ORDER BY c.name;


// ════════════════════════════════════════════════════════════════════════════════
// SECTION 4 — AGENT 4: Graph Mutator (HITL)
// Tools: kg_merge_pending(concept), kg_approve_pending(id), kg_rollback(id)
// Purpose: Create :Pending nodes for HITL review, approve or rollback.
// ════════════════════════════════════════════════════════════════════════════════

// 4a. Create a pending concept (does NOT link to curriculum until approved)
MERGE (c:Concept:Pending {id: $proposed_id})
SET c.name              = $name,
    c.description       = $description,
    c.core_theory       = $core_theory,
    c.type              = $type,
    c.color             = $color,
    c.curriculum_layer  = $layer,
    c.confidence        = $confidence,
    c.proposed_by       = "Agent3",
    c.proposed_at       = datetime(),
    c.status            = "pending";

// 4b. Connect pending node to its proposed prerequisites
MATCH (pending:Concept:Pending {id: $proposed_id}),
      (prereq:Concept {id: $prereq_id})
MERGE (prereq)-[:REQUIRES {weight: $weight, status: "pending"}]->(pending);

// 4c. SME approves: remove :Pending label, link to curriculum
MATCH (c:Concept:Pending {id: $concept_id})
REMOVE c:Pending
SET c.status = "approved", c.approved_at = datetime();

MATCH (cur:Curriculum {id: "undergrad_calculus"}), (c:Concept {id: $concept_id})
MERGE (cur)-[:COVERS {layer: c.curriculum_layer}]->(c);

// 4d. SME rolls back: detach and delete a pending node
MATCH (c:Concept:Pending {id: $concept_id})
DETACH DELETE c;

// 4e. List all pending concepts for the admin dashboard
MATCH (c:Concept:Pending)
RETURN c.id, c.name, c.type, c.confidence, c.proposed_at, c.status
ORDER BY c.proposed_at DESC;


// ════════════════════════════════════════════════════════════════════════════════
// SECTION 5 — AGENT 5: Resource Curator
// Tool: kg_attach_video(concept_id, video_data)
// Purpose: Attach a new VideoResource node found by the web crawler.
// ════════════════════════════════════════════════════════════════════════════════

// 5a. Check if video already exists (avoid duplicates)
MATCH (v:VideoResource {url: $url})
RETURN v.id, v.title;

// 5b. Attach a new ranked video to a concept
MATCH (c:Concept {id: $concept_id})
MERGE (v:VideoResource {id: $video_id})
SET v.platform    = $platform,
    v.title       = $title,
    v.url         = $url,
    v.language    = $language,
    v.duration_sec = $duration_sec,
    v.difficulty  = $difficulty,
    v.channel     = $channel,
    v.rank_score  = $rank_score,
    v.added_by    = "Agent5",
    v.added_at    = datetime()
MERGE (c)-[:HAS_RESOURCE {rank: $rank_score}]->(v);

// 5c. Get concepts that have no videos in a specific language
//     (Used by Agent 5 to prioritise crawl targets)
MATCH (c:Concept)
WHERE NOT EXISTS {
  MATCH (c)-[:HAS_RESOURCE]->(v:VideoResource {language: $lang})
}
RETURN c.id, c.name, c.curriculum_layer
ORDER BY c.curriculum_layer;

// 5d. Get all videos for a concept ordered by rank
MATCH (c:Concept {id: $concept_id})-[r:HAS_RESOURCE]->(v:VideoResource)
WHERE v.language = $lang
RETURN v.title, v.url, v.channel, v.duration_sec, v.difficulty, r.rank
ORDER BY r.rank DESC
LIMIT 3;


// ════════════════════════════════════════════════════════════════════════════════
// SECTION 6 — UI MAPPING AGENT (Phase 5)
// Progressive disclosure: 3 hover states with increasing cognitive load
// ════════════════════════════════════════════════════════════════════════════════

// 6a. HOVER STATE 1 (low CL) — formula only, no theory
//     Triggered: student hovers over a highlighted token
MATCH (c:Concept {id: $concept_id})-[:HAS_FORMULA]->(f:Formula)
WHERE f.display_level = 1 AND f.is_primary = true
RETURN c.name, c.type, c.color,
       f.latex AS formula_latex,
       f.notation_plain AS formula_plain;

// 6b. HOVER STATE 2 (medium CL) — theory explanation
//     Triggered: student clicks "explain this"
MATCH (c:Concept {id: $concept_id})
RETURN c.name, c.description, c.core_theory, c.type, c.color;

// 6c. HOVER STATE 3 (high support) — video in student's language
//     Triggered: student clicks "show me a video"
MATCH (c:Concept {id: $concept_id})-[:HAS_RESOURCE]->(v:VideoResource)
WHERE v.language = $student_lang
RETURN v.title, v.url, v.channel, v.duration_sec, v.difficulty
ORDER BY v.duration_sec
LIMIT 1;

// 6d. HOVER STATE 3 fallback — if no L1 video, return English
MATCH (c:Concept {id: $concept_id})-[:HAS_RESOURCE]->(v:VideoResource)
WHERE v.language IN [$student_lang, "en"]
RETURN v.title, v.url, v.language, v.channel
ORDER BY CASE v.language WHEN $student_lang THEN 0 ELSE 1 END, v.duration_sec
LIMIT 1;

// 6e. HOVER STATE 3 — use case context ("why am I learning this?")
MATCH (c:Concept {id: $concept_id})-[:APPLIED_IN]->(u:UseCase)
RETURN u.domain, u.description, u.problem_example
LIMIT 2;

// 6f. Full ui_text_blocks payload for a single concept
//     (Combines everything the Phase 5 UI mapping agent needs in one query)
MATCH (c:Concept {id: $concept_id})
OPTIONAL MATCH (c)-[:HAS_FORMULA {primary: true}]->(pf:Formula) WHERE pf.display_level = 1
OPTIONAL MATCH (c)-[:HAS_FORMULA]->(af:Formula)
OPTIONAL MATCH (c)-[:HAS_LABEL]->(l:L2Label {language_code: $student_lang})
OPTIONAL MATCH (c)-[:REQUIRES*1..4]->(prereq:Concept)
RETURN
  c.id              AS graph_node_id,
  c.name            AS concept_name,
  c.description     AS description,
  c.core_theory     AS theory,
  c.type            AS token_type,
  c.color           AS token_color,
  pf.notation_plain AS primary_formula,
  pf.latex          AS primary_formula_latex,
  collect(DISTINCT af.notation_plain) AS all_formulas,
  l.label           AS native_label,
  l.curriculum_name AS native_curriculum_name,
  l.text_direction  AS text_direction,
  collect(DISTINCT prereq.id) AS prerequisite_ids;


// ════════════════════════════════════════════════════════════════════════════════
// SECTION 7 — MULTILINGUAL AGENT (B1–B4 behaviours)
// ════════════════════════════════════════════════════════════════════════════════

// 7a. B1: KG-grounded canonical name retrieval
//     Returns the textbook-canonical term, NOT a translation
MATCH (c:Concept {id: $concept_id})-[:HAS_LABEL]->(l:L2Label {language_code: $lang})
RETURN l.label            AS canonical_label,
       l.curriculum_name  AS curriculum_name,
       l.description_l2   AS native_description,
       l.text_direction    AS direction;

// 7b. B2: RTL detection for Arabic and Hebrew
MATCH (c:Concept {id: $concept_id})-[:HAS_LABEL]->(l:L2Label {language_code: $lang})
RETURN l.text_direction AS direction,
       CASE l.text_direction WHEN "rtl" THEN true ELSE false END AS is_rtl;

// 7c. B3: Confidence gate — check if L2Label exists before attempting localisation
//     If no HAS_LABEL edge exists → fallback to English
OPTIONAL MATCH (c:Concept {id: $concept_id})-[:HAS_LABEL]->(l:L2Label {language_code: $lang})
RETURN c.name          AS english_name,
       l.label         AS native_label,
       l IS NOT NULL   AS has_native_label;

// 7d. B4: Formal register — return curriculum_name (textbook term) not label
//     (label may be colloquial; curriculum_name is always the formal textbook term)
MATCH (c:Concept {id: $concept_id})-[:HAS_LABEL]->(l:L2Label {language_code: $lang})
RETURN l.curriculum_name AS formal_term,
       l.label           AS common_term,
       l.text_direction  AS direction;


// ════════════════════════════════════════════════════════════════════════════════
// SECTION 8 — ADMINISTRATION AND MONITORING
// ════════════════════════════════════════════════════════════════════════════════

// 8a. Full KG statistics dashboard
MATCH (n)
RETURN labels(n)[0] AS node_label, COUNT(n) AS count
ORDER BY count DESC;

MATCH ()-[r]->()
RETURN type(r) AS relation_type, COUNT(r) AS count
ORDER BY count DESC;

// 8b. Coverage report — which concepts have which node types attached
MATCH (c:Concept)
OPTIONAL MATCH (c)-[:HAS_FORMULA]->(f:Formula)
OPTIONAL MATCH (c)-[:HAS_LABEL]->(l:L2Label {language_code: "si"})
OPTIONAL MATCH (c)-[:HAS_RESOURCE]->(v:VideoResource {language: "en"})
OPTIONAL MATCH (c)-[:APPLIED_IN]->(u:UseCase)
RETURN c.id,
       c.curriculum_layer AS layer,
       f IS NOT NULL AS has_formula,
       l IS NOT NULL AS has_sinhala_label,
       v IS NOT NULL AS has_en_video,
       u IS NOT NULL AS has_use_case
ORDER BY layer, c.id;

// 8c. Find concepts with no incoming REQUIRES edges (true root concepts)
MATCH (c:Concept)
WHERE NOT ()-[:REQUIRES]->(c)
RETURN c.id, c.name, c.curriculum_layer
ORDER BY c.curriculum_layer;

// 8d. Find concept orphans (no edges of any kind)
MATCH (c:Concept)
WHERE NOT (c)-[]-()
RETURN c.id, c.name;

// 8e. Longest prerequisite chain in the graph
MATCH path = (root:Concept)-[:REQUIRES*]->(leaf:Concept)
WHERE NOT ()-[:REQUIRES]->(root) AND NOT (leaf)-[:REQUIRES]->()
RETURN root.name AS start, leaf.name AS end, length(path) AS chain_length
ORDER BY chain_length DESC
LIMIT 5;


In [48]:
def get_prerequisite_chain(concept_id: str, depth: int = 4) -> List[Dict[str, Any]]:
    """
    Queries Neo4j to get the full transitive prerequisite chain for a given concept.

    Args:
        concept_id: The ID of the concept for which to find prerequisites.
        depth: The maximum depth of the REQUIRES relationship to traverse.

    Returns:
        A list of dictionaries, where each dictionary represents a prerequisite concept
        with its ID, name, type, color, and curriculum layer.
    """
    if not driver:
        print("Neo4j driver is not initialized.")
        return []

    cypher_query = f"""
        MATCH (c:Concept {{id: $concept_id}})-[:REQUIRES*1..{depth}]->(p:Concept)
        RETURN DISTINCT p.id      AS prereq_id,
                        p.name    AS prereq_name,
                        p.type    AS prereq_type,
                        p.color   AS prereq_color,
                        p.curriculum_layer AS layer
        ORDER BY layer, prereq_name
    """
    prerequisites = []
    with driver.session() as session:
        result = session.run(cypher_query, concept_id=concept_id)
        for record in result:
            prerequisites.append({
                "id": record["prereq_id"],
                "name": record["prereq_name"],
                "type": record["prereq_type"],
                "color": record["prereq_color"],
                "curriculum_layer": record["layer"]
            })
    return prerequisites

print("✓ `get_prerequisite_chain` tool created.")

✓ `get_prerequisite_chain` tool created.


This `get_prerequisite_chain` function is now available to query the Neo4j database for prerequisite concepts. Next, you would integrate this into your LangGraph agent. For example, you could modify the `kg_retrieval_node` to use this tool when a concept is found in the KG.

This code snippet executes two queries related to the 'related_rates' concept. The first one is the exact query from the failing verification test (T05), which counts the distinct prerequisites. The second query (2b from the reference library) lists the actual prerequisite concepts and their details up to 4 hops away. This should help you understand why the count was less than 7.

You can modify the `cypher_query` variable in the Python cell above with any other query from the `d8335334` markdown cell, or write your own custom queries to explore the graph.

In [ ]:
from typing import TypedDict, List, Dict, Any, Optional
import json
import re
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# ==========================================
# 1. DEFINE THE GLOBAL STATE
# ==========================================
class UnpackGraphState(TypedDict):
    """
    The shared memory passed between all agents in the Neuro-Symbolic pipeline.
    """
    user_input: str
    extracted_variables: Dict[str, Any]
    target_concept_id: str
    kg_context: Optional[Dict[str, Any]]
    prerequisites: List[Dict[str, Any]]
    knowledge_gap_found: bool
    svg_code: str
    messages: List[str]

# ==========================================
# 2. REAL KG TOOL (Neo4j)
# ==========================================
def get_kg_prereqs(concept_id: str) -> List[Dict[str, Any]]:
    """Tool: Queries Neo4j for the prerequisite chain (Fixing directionality)."""
    if not driver:
        return []
    
    # We use the inbound arrow <-[:REQUIRES]- to find what the concept NEEDS
    query = """
    MATCH (c:Concept {id: $concept_id})<-[:REQUIRES*1..3]-(p:Concept)
    RETURN DISTINCT p.id as id, p.name as name, p.type as type, p.color as color, p.curriculum_layer as layer
    ORDER BY layer
    """
    prereqs = []
    with driver.session() as session:
        result = session.run(query, concept_id=concept_id)
        for record in result:
            prereqs.append({
                "id": record["id"],
                "name": record["name"],
                "type": record["type"],
                "color": record["color"],
                "layer": record["layer"]
            })
    return prereqs

# ==========================================
# 3. AGENT NODES (LLM Powered)
# ==========================================

def primary_reasoning_agent(state: UnpackGraphState) -> Dict[str, Any]:
    """Agent 1: Extracts math variables and the KG-compliant Concept ID."""
    print("🤖 Parsing problem with Gemini...")
    
    # SYSTEM PROMPT for structural mapping
    prompt = f"""
    Analyze this math problem: "{state['user_input']}"
    
    1. Identify the core mathematical concept as a single machine-readable slug.
       IMPORTANT: You MUST map it to one of these known KG IDs if relevant: 
       'related_rates', 'chain_rule', 'optimization', 'limits', 'differentiation_basics'.
       For ladder/tank problems, use 'related_rates'.
       
    2. Extract known variables (e.g., L=10, dx/dt=1, x=6) and the goal variable (e.g., dy/dt).
    
    Return ONLY a JSON object. No prose.
    Format: {{"concept_id": "slug", "variables": {{"name": "value", ...}}, "goal": "target"}}
    """
    
    response = gemini_model.generate_content(prompt)
    
    # Clean response to handle markdown code blocks
    text = response.text.strip()
    if "```json" in text:
        text = text.split("```json")[1].split("```")[0].strip()
    elif "```" in text:
        text = text.split("```")[1].split("```")[0].strip()
        
    try:
        data = json.loads(text)
    except Exception as e:
        print(f"Error parsing Gemini response: {e}")
        data = {"concept_id": "unknown", "variables": {}}
    
    return {
        "target_concept_id": data.get("concept_id", "unknown"),
        "extracted_variables": data.get("variables", {}),
        "messages": [f"Target Concept: {data.get('concept_id')}"]
    }

def kg_retrieval_node(state: UnpackGraphState) -> Dict[str, Any]:
    """Programmatic Node: Hits the real Neo4j KG and fetches prerequisites."""
    concept_id = state['target_concept_id']
    print(f"📊 Searching KG for: '{concept_id}'")
    
    if concept_id == "unknown":
        return {"knowledge_gap_found": True, "prerequisites": []}
        
    prereqs = get_kg_prereqs(concept_id)
    
    if prereqs:
        print(f"✓ Found {len(prereqs)} prerequisites.")
        return {
            "prerequisites": prereqs,
            "knowledge_gap_found": False
        }
    else:
        print("⚠️ No prerequisites found in KG.")
        return {"knowledge_gap_found": True, "prerequisites": []}

def svg_generation_agent(state: UnpackGraphState) -> Dict[str, Any]:
    """Agent 3: Generates an animated SVG mind-map based on variables + KG prerequisites."""
    print("🎨 Generating animated SVG visualization...")
    
    prereqs = state.get('prerequisites', [])
    variables = state.get('extracted_variables', {})
    concept = state.get('target_concept_id', 'Unknown')
    
    prompt = f"""
    Generate the source code for a high-quality SVG math visualization.
    
    Context:
    - Main Concept: {concept}
    - Variables: {variables}
    - Found Prerequisites: {[p['name'] for p in prereqs]}
    
    Requirements:
    1. Visual schema: Draw a stylized right triangle (ladder against wall).
    2. Label the sides with the variables provided.
    3. Add a pulse animation using <animate> on the 'goal' variable.
    4. Include a sidebar or small nodes for the prerequisites found.
    5. Ensure valid SVG XML. No prose.
    """
    
    response = gemini_model.generate_content(prompt)
    svg_text = response.text.strip()
    if "<svg" in svg_text:
        # Extract everything between <svg and </svg>
        match = re.search(r'(<svg.*</svg>)', svg_text, re.DOTALL)
        svg_code = match.group(0) if match else "<svg><text y='20'>Parse Error</text></svg>"
    else:
        svg_code = "<svg><text y='20'>No SVG generated by LLM</text></svg>"
    
    return {
        "svg_code": svg_code,
        "messages": ["SVG Visualization Generated."]
    }

# ==========================================
# 4. COMPILE THE REAL GRAPH
# ==========================================

workflow = StateGraph(UnpackGraphState)

workflow.add_node("primary_reasoning_agent", primary_reasoning_agent)
workflow.add_node("kg_retrieval_node", kg_retrieval_node)
workflow.add_node("svg_generation_agent", svg_generation_agent)

workflow.add_edge(START, "primary_reasoning_agent")
workflow.add_edge("primary_reasoning_agent", "kg_retrieval_node")
workflow.add_edge("kg_retrieval_node", "svg_generation_agent")
workflow.add_edge("svg_generation_agent", END)

memory = MemorySaver()
unpack_app = workflow.compile(checkpointer=memory)

print("🚀 Unpack Multi-Agent System Compiled with Robust JSON Parsing.")


### 1. Initialize Gemini API

First, make sure your Gemini API key is set up. You can retrieve it from Colab secrets and initialize the Gemini model.

In [50]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

# Access your API key from Colab secrets
GOOGLE_API_KEY=''
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the Gemini API model
gemini_model = genai.GenerativeModel('gemini-1.5-flash') # Using a fast model for demonstration
print("✓ Gemini model initialized.")

✓ Gemini model initialized.


### 2. (Conceptual) Update Agent Functions to Use Gemini

You would modify functions like `primary_reasoning_agent` and `verification_agent` to use the `gemini_model` to generate their responses instead of mocked data. For example:

```python
def primary_reasoning_agent(state: UnpackGraphState) -> UnpackGraphState:
    print("🤖 Primary Agent: Parsing user input with Gemini...")
    # Example: Use Gemini to extract variables and target concept
    prompt = f"Extract the main mathematical concept and any relevant variables from the following problem: {state['user_input']}. Respond in JSON format with 'concept' and 'variables'."
    response = gemini_model.generate_content(prompt)
    parsed_response = json.loads(response.text) # Assuming JSON output
    
    return {
        "extracted_variables": parsed_response['variables'],
        "target_concept": parsed_response['concept'],
        "messages": state.get("messages", []) + ["Primary Agent parsed input with Gemini."]
    }
```

*Note: This modification is conceptual; you would need to implement the actual LLM calls and parsing logic within those functions.*

### 3. Run the Agent Application

Now you can execute the `unpack_app` by calling its `stream` method with an initial `user_input`.

In [ ]:
from IPython.display import display, SVG as SVGDisplay

# Define a real Calculus Problem to test the KG and Agent chain
calculus_problem = "A ladder 10 ft long rests against a vertical wall. If the bottom of the ladder slides away from the wall at a rate of 1 ft/s, how fast is the top of the ladder sliding down the wall when the bottom of the ladder is 6 ft from the wall?"

initial_state = {
    "user_input": calculus_problem,
    "messages": [],
    "extracted_variables": {},
    "target_concept_id": "",
    "kg_context": None,
    "prerequisites": [],
    "knowledge_gap_found": False,
    "svg_code": ""
}

config = {"configurable": {"thread_id": "math_problem_01"}}

print(f"🚀 Running Unpack Agent for: \n'{calculus_problem}'\n")

# Execute the workflow
for output in unpack_app.stream(initial_state, config=config):
    # Print the output from the specific node that just finished
    for node_name, state_update in output.items():
        print(f"--- Node Complete: {node_name} ---")
        if 'messages' in state_update and state_update['messages']:
            print(f"   Log: {state_update['messages'][-1]}")
        
# Final State Retrieval
final_state = unpack_app.get_state(config=config).values

print("\n" + "="*30)
print("  Graph Execution Results")
print("="*30)
print(f"Target Concept: {final_state.get('target_concept_id')}")
print(f"Variables: {final_state.get('extracted_variables')}")
print(f"Prerequisite Count: {len(final_state.get('prerequisites', []))}")

# Display the Generated SVG
if final_state.get('svg_code'):
    print("\n--- Displaying Animated Math Visualization ---")
    display(SVGDisplay(final_state['svg_code']))
else:
    print("No SVG generated.")


🚀 Starting the Unpack Agent Application...
🤖 Primary Agent: Parsing user input...
{'primary_reasoning_agent': {'extracted_variables': {'object': 'ladder', 'length': 10}, 'target_concept': 'Related Rates', 'messages': ['Primary Agent parsed input.']}}
📊 KG Node: Querying Neo4j for 'Related Rates'...
⚠️ Knowledge Gap Identified!
{'kg_retrieval_node': {'kg_context': None, 'knowledge_gap_found': True}}
⚖️ Verification Agent: Evaluating new concept...
{'verification_agent': {'verification_score': 85, 'draft_cypher': "MERGE (c:Concept {name: 'Related Rates'})", 'messages': ['Primary Agent parsed input.', 'Verification scored: 85/100.']}}
{'__interrupt__': ()}
✅ Application run complete.

Final State:
{'user_input': 'Water drains from a conical tank at 2 m³/min. The tank has height 4 m and radius 2 m. How fast is the water level dropping when h = 3 m?', 'extracted_variables': {'object': 'ladder', 'length': 10}, 'target_concept': 'Related Rates', 'kg_context': None, 'knowledge_gap_found': True